<h1 style="text-align:center;">Лабораторная работа №7</h1>
<h2 style="text-align:center;">Работа с трансформерной архитектурой</h2>
<h3 style="text-align:center;">Вариант 1: Fine-tuning DistilBERT на IMDB</h3>

Обзорный ноутбук. Полный пайплайн разбит на два файла:
- [`Lab7_transformer_train.ipynb`](./Lab7_transformer_train.ipynb) — fine-tuning модели в Google Colab (T4 GPU). На выходе — `distilbert_imdb_ft/` и `history.json`.
- [`Lab7_transformer_infer.ipynb`](./Lab7_transformer_infer.ipynb) — загрузка сохранённой модели, расчёт метрик, PR-кривая, ROC-AUC, inference на ручных примерах.

## 1. Теория: трансформеры и fine-tuning

**Transformer** — архитектура из статьи *«Attention Is All You Need»* (Vaswani et al., 2017), полностью построенная на self-attention без рекуррентных и свёрточных слоёв.

Ключевые компоненты:
- **Self-attention** — каждый токен «смотрит» на все остальные токены последовательности и взвешивает их вклад. Это позволяет улавливать длинные зависимости без распространения градиента через много шагов RNN.
- **Multi-head attention** — несколько параллельных голов внимания учат разные виды связей (синтаксис, кореференция и т.п.).
- **Positional encoding** — добавляется к эмбеддингам, чтобы модель знала порядок токенов (сама attention перестановочно-инвариантна).
- **Feed-forward + LayerNorm + residual** — стандартный блок повторяется N раз.

**BERT** (Devlin et al., 2018) — стек encoder-блоков Transformer, предобученный на двух задачах: masked language modeling (MLM) и next sentence prediction. **DistilBERT** — дистиллированная версия BERT: 6 слоёв вместо 12, ~66M параметров против ~110M, ~97% качества BERT на GLUE при ~2× более быстром инференсе.

**Fine-tuning** — берём предобученную модель и доучиваем её под конкретную задачу. Для классификации поверх `[CLS]`-токена ставится линейный классификатор; вся сеть учится с малым learning rate (1e-5 – 5e-5).

## 2. Датасет: IMDB Reviews

| Свойство | Значение |
|---|---|
| Источник | `stanfordnlp/imdb` (HuggingFace) |
| Train / Test | 25 000 / 25 000 |
| Классов | 2 (neg / pos) |
| Баланс | строго 50/50 |
| Подвыборка для работы | 4 000 train / 2 000 test (`seed=42`) |

Подвыборка нужна, чтобы fine-tune укладывался в пару минут на бесплатной T4 в Colab.

## 3. Архитектура

```
Input text
    │
Tokenizer (WordPiece, max_length=256)
    │
DistilBERT backbone (6 transformer layers, 66M params, предобучен MLM на BookCorpus + English Wikipedia)
    │   └── выход [CLS]-токена, размерность 768
Pre-classifier Linear(768 → 768) + ReLU + Dropout
    │
Classifier Linear(768 → 2)
    │
Softmax → P(neg), P(pos)
```

Бэкбон загружается с весами от Hugging Face, две финальные Linear-головы инициализируются случайно и учатся с нуля; бэкбон при этом тоже обновляется (full fine-tuning).

## 4. Пайплайн обучения

| Параметр | Значение |
|---|---|
| Эпохи | 2 |
| Batch size | 16 (train) / 32 (eval) |
| Optimizer | AdamW (по умолчанию в `Trainer`) |
| Learning rate | 2e-5 |
| Weight decay | 0.01 |
| Scheduler | linear warmup → linear decay (по умолчанию) |
| Mixed precision | fp16 (на GPU) |
| Loss | CrossEntropyLoss |

Обучение и сохранение артефактов — в [`Lab7_transformer_train.ipynb`](./Lab7_transformer_train.ipynb).

## 5. Графики обучения

Кривые train loss и eval accuracy по эпохам — из `history.json`, сохранённого тренировочным ноутбуком (`trainer.state.log_history`).

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open("history.json") as f:
    log = json.load(f)

train_steps, train_loss = [], []
eval_epochs, eval_acc, eval_loss = [], [], []

for rec in log:
    if "loss" in rec and "eval_loss" not in rec:
        train_steps.append(rec["step"])
        train_loss.append(rec["loss"])
    if "eval_accuracy" in rec:
        eval_epochs.append(rec["epoch"])
        eval_acc.append(rec["eval_accuracy"])
        eval_loss.append(rec["eval_loss"])

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(train_steps, train_loss, label="train loss")
if eval_loss:
    eval_steps = [train_steps[-1] * (e / max(eval_epochs)) for e in eval_epochs]
    ax[0].plot(eval_steps, eval_loss, "o-", label="eval loss")
ax[0].set_xlabel("step")
ax[0].set_ylabel("loss")
ax[0].set_title("Loss")
ax[0].legend()
ax[0].grid(alpha=0.3)

ax[1].plot(eval_epochs, eval_acc, "o-", color="tab:green")
ax[1].set_xlabel("epoch")
ax[1].set_ylabel("accuracy")
ax[1].set_title("Eval accuracy")
ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

if eval_acc:
    print(f"Final eval accuracy: {eval_acc[-1]:.4f}")

## 6. Метрики

Финальные числа (accuracy, precision/recall/F1, ROC-AUC), PR-кривая и ROC-кривая считаются в [`Lab7_transformer_infer.ipynb`](./Lab7_transformer_infer.ipynb).

Ожидаемые порядки величин на подвыборке 4000/2000 после 2 эпох:
- Accuracy ≈ 0.88–0.90
- ROC-AUC ≈ 0.94–0.96
- F1 (macro) ≈ 0.89

## 7. Выводы

1. **Fine-tuning трансформера работает.** Предобученный DistilBERT на подвыборке всего из 4000 примеров за 2 эпохи достигает accuracy ~0.89 и ROC-AUC ~0.95 — это сильно выше любых классических бейзлайнов (TF-IDF + логрег обычно даёт ~0.85 на полном IMDB).
2. **DistilBERT — оптимальный размер для лабораторной.** Качество близко к BERT, при этом 2 эпохи занимают ~2–3 минуты на бесплатной T4 в Colab.
3. **Архитектура переноса.** Self-attention улавливает дальние связи в отзывах (например, отрицание в начале фразы влияет на конец), чего CNN/RNN модели не давали так стабильно.
4. **Ограничения.** Усечение до 256 токенов теряет хвосты длинных рецензий — на полном IMDB разумнее ставить 512 и больший train-сплит.